# Lesson 12: Save Data in a Database (SQLite)

In Lesson 8 we saved data to a file (a `.txt` or `.json`). That works well when the data is small.

But imagine a food delivery business with **thousands** of customer reviews. Every day more come in. You want to:
- add new reviews fast,
- find all the *negative* ones,
- count how many are about *delivery*.

Doing this with a plain file becomes painful. This is the job of a **database**.

Think of a database as a **smart Excel sheet** that your program can read and write, and search through very quickly.

## The good news: SQLite is already inside Python

For most databases you have to install and run a separate server. **SQLite** is different:

- It comes **built into Python** — nothing to install (`import sqlite3` and you are ready).
- It stores everything in **one small file** (like `shop.db`).

That makes it perfect for learning, and for small real apps too.

In [ ]:
import sqlite3

# connect() opens the database file. If the file does not exist yet,
# SQLite creates it for us. So this one line makes a new database.
conn = sqlite3.connect("shop.db")

# The cursor is the tool we use to send commands to the database.
cursor = conn.cursor()

print("Database is ready!")

## Step 1: Create a table

A database holds **tables**. A table is just rows and columns, like a sheet.

We will make a `reviews` table with these columns:
- `id` — a number for each row (the database fills this in automatically)
- `text` — the review text
- `label` — positive / negative / neutral
- `score` — a number from 1 to 5

We also tell SQLite the **type** of each column: `TEXT` for words, `INTEGER` for whole numbers.

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        id INTEGER PRIMARY KEY,
        text TEXT,
        label TEXT,
        score INTEGER
    )
""")

print("Table created!")

> `IF NOT EXISTS` means: only create the table the first time. If you run this cell again, it will not complain.
>
> `id INTEGER PRIMARY KEY` is a little magic — SQLite gives each new row its own number automatically (1, 2, 3...). We never have to set the `id` ourselves.

## Step 2: Add data (write to the database)

Now let's put one review into the table.

Notice the question marks `?`. We do **not** glue the values directly into the text. We put `?` as placeholders, and pass the real values separately. This is the safe, correct way to do it.

In [ ]:
cursor.execute(
    "INSERT INTO reviews (text, label, score) VALUES (?, ?, ?)",
    ("The biryani was amazing!", "positive", 5)
)

# commit() actually saves the change to the file. Without it, nothing is saved.
conn.commit()

print("One review saved.")

Adding rows one by one is slow. To add **many** at once, use `executemany` with a list.

In [ ]:
more_reviews = [
    ("Delivery was very late and food was cold.", "negative", 1),
    ("Good taste, fair price.", "positive", 4),
    ("It was okay, nothing special.", "neutral", 3),
    ("App kept crashing during payment.", "negative", 2),
]

cursor.executemany(
    "INSERT INTO reviews (text, label, score) VALUES (?, ?, ?)",
    more_reviews
)
conn.commit()

print("Added", len(more_reviews), "more reviews.")

## Step 3: Read data (get it back out)

Writing is only half the story. Let's **read** the reviews back.

We run a `SELECT` command, then use `fetchall()` to get all the rows as a list.

In [ ]:
cursor.execute("SELECT * FROM reviews")
rows = cursor.fetchall()

for row in rows:
    print(row)

Each row comes back as a small group of values: `(id, text, label, score)`.

The real power of a database is **asking questions** with `WHERE`. For example, show only the negative reviews:

In [ ]:
cursor.execute("SELECT text, score FROM reviews WHERE label = ?", ("negative",))
negatives = cursor.fetchall()

print("Negative reviews:")
for text, score in negatives:
    print("-", text, "(score:", score, ")")

## Step 4: Close when you are done

When your program is finished with the database, close the connection. This makes sure everything is saved and tidy.

In [ ]:
conn.close()
print("Database closed.")

## Quick recap

Just four ideas and you can use a database:

| Action | Command |
|---|---|
| Open / create the DB | `sqlite3.connect("shop.db")` |
| Make a table | `CREATE TABLE ...` |
| Add data | `INSERT INTO ...` + `conn.commit()` |
| Read data | `SELECT ...` + `fetchall()` |

A `.db` file lives on your disk, so your data stays even after you close the program. That is the big difference from a normal Python list, which disappears when the program ends.

**Where we use this next:** in Project 1, instead of saving the analyzed reviews to `results.json`, we will save them into a `reviews` table exactly like this one. Now the business keeps a growing history of all feedback.

## Exercise

A small shop wants to store its products in a database.

1. Create a table called `products` with columns: `id`, `name` (TEXT), and `price` (INTEGER).
2. Add at least 3 products (for example: "Tea", 10).
3. Read and print all products.
4. **Bonus:** print only the products with a price greater than 20 (hint: `WHERE price > ?`).

Tip: you can reuse `shop.db` from above — a database can hold many tables.